# Projet

In [5]:
import pandas as pd

BASE_URL_CSV = r"C:\Users\Kainwang Roger\Desktop\data_engineer_cours_bourse\Projet_Scrapping\Books.csv\Books.csv"

df = pd.read_csv(BASE_URL_CSV)

df.head()


C:\Users\Kainwang Roger\AppData\Local\Temp\ipykernel_7056\182377300.py:5: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(BASE_URL_CSV)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
from typing import List, Optional
from pathlib import Path
from include.utils.custom_logging import setup_logging, get_logger

BASE_URL = "https://books.toscrape.com/"

BASE_URL_CSV = r"C:\Users\Kainwang Roger\Desktop\data_engineer_cours_bourse\Projet_Scrapping\Books.csv\Books.csv"
# -------------------------
# LOGGING
# -------------------------
setup_logging()
logger = get_logger(__name__)

# ------------------------- 
# PATHS
# -------------------------
BASE_DIR = Path(__file__).resolve().parent.parent.parent
DATA_DIR = BASE_DIR / "data" / "raw"
DATA_DIR.mkdir(parents=True, exist_ok=True)

LINKS_FILE = DATA_DIR / "book_links.txt"


In [ ]:

# --------------------------------------------------
# HTTP / SESSION
# --------------------------------------------------
def create_session() -> requests.Session:
    """
    Crée et configure une session HTTP.
    Séparée pour être testable et compatible Airflow.
    """
    session = requests.Session()
    session.headers.update({
        "User-Agent": "BooksETL/1.0"
    })
    return session

In [ ]:
# --------------------------------------------------
# HTTP / PARSING
# --------------------------------------------------
def get_soup(session: requests.Session, url: str) -> Optional[BeautifulSoup]:
    try:
        response = session.get(url, timeout=10)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html5lib")
    except requests.RequestException as e:
        logger.error(f"Erreur HTTP sur {url} : {e}")
        return None

In [ ]:

# --------------------------------------------------
# EXTRACTION
# --------------------------------------------------
def get_all_book_links(session: requests.Session) -> List[str]:
    logger.info("Début extraction des liens")
    book_links = []

    for page in range(1, 51):
        url = f"{BASE_URL}catalogue/page-{page}.html"
        soup = get_soup(session, url)

        if not soup:
            continue

        books = soup.select("article.product_pod h3 a")

        for book in books:
            relative_link = book["href"].replace("../../", "")
            full_link = f"{BASE_URL}catalogue/{relative_link}"
            book_links.append(full_link)

        logger.info(f"Page {page} récupérée ({len(books)} livres)")
        time.sleep(0.2)

    return book_links

In [ ]:
# --------------------------------------------------
# SAUVEGARDE
# --------------------------------------------------
def save_book_links(file_path: Path, links: List[str]) -> None:
    # links = get_all_book_links(create_session())

    with open(file_path, "w", encoding="utf-8") as f:
        for link in links:
            f.write(link + "\n")
    if links:
        logger.info(f"{len(links)} liens sauvegardés dans {file_path}")

    else:
        logger.warning("Aucun lien trouvé")


# --------------------------------------------------
# EXECUTION LOCALE (HORS AIRFLOW)
# --------------------------------------------------
def main():

    save_book_links(LINKS_FILE, links=get_all_book_links())


if __name__ == "__main__":
    main()